# FRLM Data Exploration

Explore the PMC Open Access corpus, extracted entities, relations,
temporal coverage, and router label distribution for the FRLM project.

In [ ]:
import sys
import json
from pathlib import Path
from collections import Counter

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path("__file__").resolve().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config.config import load_config

cfg = load_config()
sns.set_theme(style="whitegrid", palette="colorblind", font_scale=1.1)

CORPUS_DIR = PROJECT_ROOT / cfg.paths.corpus_dir
PROCESSED_DIR = PROJECT_ROOT / cfg.paths.processed_dir
LABELS_DIR = PROJECT_ROOT / cfg.paths.labels_dir

print(f"Project: {cfg.project.name} v{cfg.project.version}")
print(f"Corpus dir   : {CORPUS_DIR}")
print(f"Processed dir: {PROCESSED_DIR}")
print(f"Labels dir   : {LABELS_DIR}")

In [ ]:
# --- Corpus statistics ---
xml_files = list(CORPUS_DIR.glob("*.xml"))
print(f"Total XML files: {len(xml_files)}")

if xml_files:
    sizes = [f.stat().st_size / 1024 for f in xml_files]  # KB
    df_corpus = pd.DataFrame({"file": [f.name for f in xml_files], "size_kb": sizes})
    print(f"Total size: {sum(sizes)/1024:.2f} MB")
    print(f"Mean file size: {np.mean(sizes):.1f} KB")
    print(f"Median file size: {np.median(sizes):.1f} KB")
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(sizes, bins=50, edgecolor="white")
    axes[0].set_xlabel("File Size (KB)")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Corpus File Size Distribution")
    
    axes[1].boxplot(sizes, vert=True)
    axes[1].set_ylabel("File Size (KB)")
    axes[1].set_title("File Size Box Plot")
    plt.tight_layout()
    plt.show()
else:
    print("No corpus files found. Run: make download-corpus")

In [ ]:
# --- Entity distribution ---
entity_files = sorted(PROCESSED_DIR.glob("entities_*.json"))
print(f"Entity files: {len(entity_files)}")

all_entities = []
for path in entity_files[:100]:  # sample first 100
    try:
        with open(path, "r") as f:
            all_entities.extend(json.load(f))
    except (json.JSONDecodeError, OSError):
        pass

if all_entities:
    df_ent = pd.DataFrame(all_entities)
    print(f"Total entities (sampled): {len(df_ent)}")
    print(f"\nEntity type distribution:")
    print(df_ent["label"].value_counts().head(15))
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    df_ent["label"].value_counts().head(10).plot.barh(ax=axes[0], color=sns.color_palette())
    axes[0].set_title("Top 10 Entity Types")
    axes[0].set_xlabel("Count")
    
    if "confidence" in df_ent.columns:
        df_ent["confidence"].dropna().hist(bins=50, ax=axes[1], edgecolor="white")
        axes[1].set_title("Entity Confidence Distribution")
        axes[1].set_xlabel("Confidence")
        axes[1].axvline(cfg.extraction.entity.confidence_threshold, color="red",
                       ls="--", label=f"Threshold={cfg.extraction.entity.confidence_threshold}")
        axes[1].legend()
    plt.tight_layout()
    plt.show()
else:
    print("No entities found. Run: make extract-entities")

In [ ]:
# --- Relation distribution ---
relation_files = sorted(PROCESSED_DIR.glob("relations_*.json"))
print(f"Relation files: {len(relation_files)}")

all_relations = []
for path in relation_files[:100]:
    try:
        with open(path, "r") as f:
            all_relations.extend(json.load(f))
    except (json.JSONDecodeError, OSError):
        pass

if all_relations:
    df_rel = pd.DataFrame(all_relations)
    print(f"Total relations (sampled): {len(df_rel)}")
    print(f"\nRelation type distribution:")
    print(df_rel["relation_type"].value_counts())
    
    fig, ax = plt.subplots(figsize=(10, 5))
    df_rel["relation_type"].value_counts().plot.bar(ax=ax, color=sns.color_palette("viridis", 10))
    ax.set_title("Relation Type Distribution")
    ax.set_ylabel("Count")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No relations found. Run: make extract-relations")

In [ ]:
# --- Label distribution (retrieval vs. generation) ---
label_files = sorted(LABELS_DIR.glob("labels_*.json"))
print(f"Label files: {len(label_files)}")

retrieval_ratios = []
valid_count = 0
invalid_count = 0

for path in label_files[:200]:
    try:
        with open(path, "r") as f:
            data = json.load(f)
        if data.get("valid", False):
            valid_count += 1
        else:
            invalid_count += 1
        
        labels = data.get("labels", [])
        retrieval_chars = sum(l.get("end", 0) - l.get("start", 0)
                             for l in labels if l.get("label") == "RETRIEVAL")
        total_chars = sum(l.get("end", 0) - l.get("start", 0) for l in labels)
        if total_chars > 0:
            retrieval_ratios.append(retrieval_chars / total_chars)
    except (json.JSONDecodeError, OSError):
        pass

if retrieval_ratios:
    print(f"Valid: {valid_count}, Invalid: {invalid_count}")
    print(f"Mean retrieval ratio: {np.mean(retrieval_ratios):.3f}")
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].hist(retrieval_ratios, bins=40, edgecolor="white", color="#3498db")
    axes[0].axvline(cfg.labeling.validation.min_retrieval_ratio, color="red", ls="--",
                   label=f"Min={cfg.labeling.validation.min_retrieval_ratio}")
    axes[0].axvline(cfg.labeling.validation.max_retrieval_ratio, color="red", ls="--",
                   label=f"Max={cfg.labeling.validation.max_retrieval_ratio}")
    axes[0].set_xlabel("Retrieval Token Ratio")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Retrieval vs Generation Token Ratio")
    axes[0].legend()
    
    axes[1].pie([valid_count, invalid_count], labels=["Valid", "Invalid"],
               autopct="%1.1f%%", colors=["#2ecc71", "#e74c3c"])
    axes[1].set_title("Label Quality")
    plt.tight_layout()
    plt.show()
else:
    print("No labels found. Run: make label-data")

In [ ]:
# --- Summary statistics ---
summary = {
    "Corpus files": len(xml_files),
    "Entity files": len(entity_files),
    "Relation files": len(relation_files),
    "Label files": len(label_files),
    "Valid labels": valid_count if label_files else "N/A",
    "Entities (sampled)": len(all_entities),
    "Relations (sampled)": len(all_relations),
    "Mean retrieval ratio": f"{np.mean(retrieval_ratios):.3f}" if retrieval_ratios else "N/A",
    "Corpus size (MB)": f"{sum(sizes)/1024:.2f}" if xml_files else "0",
}

df_summary = pd.DataFrame(list(summary.items()), columns=["Metric", "Value"])
print("=" * 50)
print("FRLM Data Exploration Summary")
print("=" * 50)
print(df_summary.to_string(index=False))